# 02 · CLIP heads (A1–A4) and all head experiments
P1 (GenImage protocol), bias ablation, LOGO, data efficiency, few-shot, layer probes, shuffled-label sanity check, t-SNE.

In [ ]:
# --- Kaggle setup (run first in every notebook) ---
# Settings: Accelerator = GPU (T4 x2 or P100), Internet = ON.
# Add Data: vtphatt2/GenImage-stable-diffusion-v1-4 and vtphatt2/GenImage-Validation
#           (+ the output dataset of the previous notebook, if you split stages across sessions).
import os, subprocess
REPO = "https://github.com/<your-team>/fakelens.git"   # <- change, or upload the repo as a Kaggle dataset
if not os.path.exists("/kaggle/working/fakelens"):
    subprocess.run(["git", "clone", REPO, "/kaggle/working/fakelens"], check=True)
os.chdir("/kaggle/working/fakelens")
!pip install -q open_clip_torch
!pip install -q -e . --no-deps
!ls /kaggle/input

In [ ]:
# Check the dataset folder names printed above and fix them here if they differ.
CFG = "configs/default.yaml"
SET = "--set roots.train_root=/kaggle/input/genimage-stable-diffusion-v1-4 roots.test_root=/kaggle/input/genimage-validation"
RUNS = "/kaggle/working/runs"

In [ ]:
for m in ['A1', 'A2', 'A3', 'A4']:
    !python -m fakelens.train_head --config $CFG $SET --model {m} --experiment p1

In [ ]:
!python -m fakelens.train_head --config $CFG $SET --model A2 --experiment p1 --raw
!python -m fakelens.train_head --config $CFG $SET --model A2 --experiment shuffle

In [ ]:
for m in ['A1', 'A2']:
    for e in ['logo', 'data_eff', 'fewshot']:
        !python -m fakelens.train_head --config $CFG $SET --model {m} --experiment {e}

In [ ]:
for m in ['A2', 'A3', 'A4']:
    !python -m fakelens.train_head --config $CFG $SET --model {m} --experiment layer_probe --seeds 0

In [ ]:
!python -m fakelens.analysis.embeddings --config $CFG $SET --model A2 --ckpt $RUNS/p1/A2/seed0/head.pt
!python -m fakelens.evaluate aggregate --runs $RUNS/p1 --out results/p1
print(open('results/p1/summary.md').read())